In many Q&A application we want to allow the user to have a back and forth coversation, meaning the application needs some "memory" of past questions and answers, and some logic for incorporating those into its current thinking .

Approches:


*   Chains, in which we always execute a retrieval step;
*   Agents, in which we give an LLM discretion over whether and how to execute a retrieval step (or multiple step).



In [1]:
!pip install langchain-groq langchain-classic langchain-huggingface langchain-chroma langchain-community

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")
Hf_token=os.getenv('Hf_Token')

from langchain_groq import ChatGroq
llm=ChatGroq(groq_api_key=groq_api_key, model ="llama-3.1-8b-instant")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7d3b68733c80>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7d3b685cdc40>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Updated import for LangChain v1.0+
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


#load,chunk and index the content of the blog to create a retriever
import bs4
loader=WebBaseLoader(
    web_path=("https://www.cloudjournee.com/blog/multi-agent-ai-on-amazon-bedrock-a-practical-guide-for-enterprise-use-cases/"),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content","post-title","post-header")
        )
    ),
)
docs=loader.load()
docs

/tmp/ipykernel_49220/583530403.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


[Document(metadata={'source': 'https://www.cloudjournee.com/blog/multi-agent-ai-on-amazon-bedrock-a-practical-guide-for-enterprise-use-cases/'}, page_content='\nMost enterprise AI pilots fail for a reason that has nothing to do with the model.\nThe model works. The demo impresses. Then the project tries to scale across the organisation — and stalls.\nThe reason is structural. One AI assistant cannot realistically understand every department, every workflow, every data source, and every business rule inside a large enterprise. Finance behaves differently from supply chain. Compliance differs from customer operations. Internal knowledge is fragmented across CRMs, ERPs, tickets, PDFs, and legacy databases that nobody owns end to end.\nThis is where multi-agent AI changes the equation.\nInstead of one overloaded assistant trying to do everything, enterprises can deploy multiple specialised agents that collaborate. One retrieves data. Another validates policies. A third generates recommenda

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits=splitter.split_documents(docs)

In [5]:
len(splits)

14

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding=HuggingFaceEmbeddings(model="all-MiniLm-L6-v2")
embedding

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLm-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HuggingFaceEmbeddings(model_name='all-MiniLm-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [7]:
from langchain_chroma import Chroma

vectorstore=Chroma.from_documents(docs,embedding=embedding)
vectorstore

In [8]:
#querying with vectorstore
response=vectorstore.similarity_search_with_score("what did i asked you earlier ..?")
# response


In [9]:
retriever=vectorstore.as_retriever()
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x7d3b29736f30>, search_kwargs={})

In [10]:
# Chatprompt template
from langchain_core.prompts import ChatPromptTemplate

system_prompt=(
        'you are an assistant for question-answering tasks.'
        "use the following pieces of retrieved context to answer"
        "the question.If you dont know the answer , say that you dont konw."
        "Use three sentence maximum and keep the answer concise."
        "{context}"
    )
prompt=ChatPromptTemplate.from_messages(

    [
        ("system",system_prompt),
        ("human","{input}"),
    ]
)



In [11]:
from langchain_classic.chains import create_retrieval_chain

question_answer=create_stuff_documents_chain(llm,prompt)
rag_chain=create_retrieval_chain(retriever,question_answer )
# response = question_answer.invoke({"input": "Hows you ?", "context": []})
# print(response)
response = rag_chain.invoke({"input": "what does cloudJournee do ..?"})
response

{'input': 'what does cloudJournee do ..?',
 'context': [Document(id='0c006ad8-a46a-446d-8e67-a64232b59fbc', metadata={'source': 'https://www.cloudjournee.com/blog/multi-agent-ai-on-amazon-bedrock-a-practical-guide-for-enterprise-use-cases/'}, page_content='\nMost enterprise AI pilots fail for a reason that has nothing to do with the model.\nThe model works. The demo impresses. Then the project tries to scale across the organisation — and stalls.\nThe reason is structural. One AI assistant cannot realistically understand every department, every workflow, every data source, and every business rule inside a large enterprise. Finance behaves differently from supply chain. Compliance differs from customer operations. Internal knowledge is fragmented across CRMs, ERPs, tickets, PDFs, and legacy databases that nobody owns end to end.\nThis is where multi-agent AI changes the equation.\nInstead of one overloaded assistant trying to do everything, enterprises can deploy multiple specialised age

In [12]:
response['answer']

'CloudJournee is an AWS Advanced Tier Partner with the AWS AI Competency that designs, builds, and operates production-grade multi-agent AI systems on Amazon Bedrock for enterprise use cases. They help organizations implement multi-agent AI solutions for customer operations, knowledge intelligence, healthcare, and workflow automation.'

In [13]:
# response=question_answer.invoke({'input':"what did i asked you earlier ","context":[]})
# print(response)

Adding ChatHistory for conversational QA

In [14]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

contextual_system_prompt=(
    """
    Given a chat history and the latest user question,
    which migh reference context in the chat history,
    formulate a standalone wuestion which can be understood,
    without the user chat history , dont answer any question
    instead just reply with ,I Don't know , its wasnt found in
    conversationaal history
    """
)

contextual_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",contextual_system_prompt),
        ("human","{input}"),
        MessagesPlaceholder('chat_history')
    ]
)

In [15]:
history_aware_retriever=create_history_aware_retriever(llm,retriever,contextual_prompt )
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x7d3b29736f30>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChu

In [21]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        ("human","{input}"),
        MessagesPlaceholder('chat_history')
    ]
)

question_answer_chain=create_stuff_documents_chain(llm,qa_prompt)
rag_chain= create_retrieval_chain(history_aware_retriever,question_answer_chain)

from langchain_core.messages import AIMessage,HumanMessage
chat_history=[]
# question="what is CLoudJourness Mission and Vision..?"
question="what is CLoudJourness ..?"
response1=rag_chain.invoke({"input":question,"chat_history":chat_history})

chat_history.extend([
    HumanMessage(content=question),
    AIMessage(content=response1['answer'])
])

question2 = "tell me more about it ..?"
response2=rag_chain.invoke({"input":question2,"chat_history":chat_history})
print(response2['answer'])

 

They have expertise in delivering multi-agent AI systems across various industries, including customer operations, knowledge intelligence, healthcare, and workflow automation.

They are mentioned in the retrieved context as a solution provider for enterprises looking to implement multi-agent AI on Amazon Bedrock.


In [22]:
print(response1['answer'])

CloudJournee is an AWS Advanced Tier Partner with the AWS AI Competency that provides services for designing, building, and operating production-grade multi-agent AI systems on Amazon Bedrock for enterprise use cases.
